# Project 1: Phân tích tình hình nền kinh tế Việt Nam
## Đề Tài: Sự chuyển dịch cơ cấu giữa các khu vực kinh tế gắn liền với chỉ số TFP - dự báo định hướng phát triển quốc gia

Dự án này đảm bảo thực hiện những mục tiêu sau bao gồm về lý thuyết và kỹ thuật:
- Lý thuyết: Phân tích sự tương tác và vai trò của 3 khu vực kinh tế (Khu vực I: Nông lâm thủy sản; Khu vực II: Công nghiệp & Xây dựng; Khu vực III: Dịch vụ).
- Kỹ thuật:
  - Thực hiện hợp lý phương pháp xử lý dữ liệu đối với bài toán cụ thể đang xét
  - Khám phá dữ liệu → nắm bắt được quy luật, xu hướng của dữ liệu
  - Phân tích chuỗi thời gian
  - Thực hiện mô tả thống kê bằng mô hình học máy
  - Trực quan hóa và đánh giá kết quả

### Bước 1: Tích hợp dữ liệu
Nhóm đã thu thập các files liên quan đến bài toán và đính kèm trong drive chung. Thực hiện tiến hành load các files đó vào dataframe sử dụng thư viện pandas.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
df_gdp_region = pd.read_excel('/content/datasets/Co_cau_GDP_theo_khu_vuc_Full.xlsx')
df_tfp = pd.read_excel('/content/datasets/Dong_gop_TFP_Full.xlsx')
df_gdp = pd.read_excel('/content/datasets/Su_dung_GDP_Full.xlsx')
df_gni = pd.read_excel('/content/datasets/Tong_thu_nhap_quoc_gia_GNI_Full.xlsx')
df_wb = pd.read_csv('/content/datasets/world_bank.csv')

In [ ]:
# ============================================================
# TIỀN XỬ LÝ DỮ LIỆU: df_gdp (Su_dung_GDP_Full.xlsx)
# Đề tài: Chuyển dịch cơ cấu kinh tế & TFP - Việt Nam
# ============================================================
# Các bước thực hiện:
#   1. Tích hợp dữ liệu (Data Integration)
#   2. Làm sạch dữ liệu (Data Cleaning)
#   3. Trích lọc dữ liệu (Data Selection)
#   4. Chuyển đổi dữ liệu (Data Transformation)
# ============================================================

import pandas as pd
import numpy as np

# ─────────────────────────────────────────────────────────────
# BƯỚC 1 – TÍCH HỢP: Đọc file gốc (không đặt header tự động)
# ─────────────────────────────────────────────────────────────
df_raw = pd.read_excel(
    '/content/datasets/Su_dung_GDP_Full.xlsx',
    header=None   # đọc toàn bộ bao gồm cả hàng tiêu đề gốc
)

print("=== KIỂM TRA CẤU TRÚC GỐC ===")
print(f"Shape gốc: {df_raw.shape}")   # (21 hàng × 32 cột)

# ─────────────────────────────────────────────────────────────
# BƯỚC 2 – LÀM SẠCH
#   a) Đổi tên cột năm: "So b  2024" → "2024"
#   b) Thay giá trị ".." (thiếu) → NaN
# ─────────────────────────────────────────────────────────────

# a) Xử lý header năm từ hàng 0, bắt đầu từ cột 2
year_headers = []
for y in df_raw.iloc[0, 2:]:
    y_str = str(y).strip()
    if 'So b' in y_str:          # "So b  2024" → "2024"
        year_headers.append('2024')
    else:
        year_headers.append(y_str.split('.')[0])   # bỏ phần ".0" nếu có

print("\n=== HEADER NĂM SAU XỬ LÝ ===")
print(year_headers)

# ─────────────────────────────────────────────────────────────
# BƯỚC 3 – TRÍCH LỌC: Chỉ giữ các hàng "Giá trị (tỷ đồng)"
#   Drop toàn bộ:
#     - Tất cả hàng "Cơ cấu (%)"  (hàng 11–20)
#     - Cột "Chỉ tiêu" (col 0) và "Loại" (col 1) – dùng làm tên cột xong thì bỏ
#     - Hàng "Giá trị (tỷ đồng) | Tổng tích lũy tài sản" (hàng 10 – chứa '..')
# ─────────────────────────────────────────────────────────────

# Mapping: vị trí hàng gốc → tên cột mới (tiếng Anh, snake_case)
keep_rows = {
    1: 'GDP_TongSo',                # TỔNG SỐ GDP theo phương pháp sử dụng
    3: 'TongTaiSanCoDinh',          # Tổng tài sản cố định
    4: 'ThayDoiTonKho',             # Thay đổi tồn kho
    5: 'TieuDungCuoiCung',          # Tiêu dùng cuối cùng (tổng)
    6: 'TieuDungNhaNuoc',           # Tiêu dùng cuối cùng – Nhà nước
    7: 'TieuDungHoDanCu',           # Tiêu dùng cuối cùng – Hộ dân cư
    8: 'ChanLechXuatNhapKhau',      # Chênh lệch xuất – nhập khẩu hàng hóa & DV
    9: 'SaiSo',                     # Sai số thống kê
}

# ─────────────────────────────────────────────────────────────
# BƯỚC 4 – CHUYỂN ĐỔI
#   a) Dùng .T (transpose): cột thời gian → hàng (mỗi hàng = 1 năm)
#   b) Thay '..' → NaN; ép kiểu về float/int
#   c) Đặt lại tên cột rõ ràng
# ─────────────────────────────────────────────────────────────

# Xây dựng dict từng cột
data_dict = {'Nam': [int(y) for y in year_headers]}

for row_idx, col_name in keep_rows.items():
    raw_values = df_raw.iloc[row_idx, 2:].tolist()
    # b) Làm sạch: '..' → NaN
    cleaned = [
        np.nan if str(v).strip() == '..' else float(v)
        for v in raw_values
    ]
    data_dict[col_name] = cleaned

# Tạo DataFrame sạch (đây chính là df_gdp sau tiền xử lý)
df_gdp = pd.DataFrame(data_dict)

# Đặt 'Nam' làm index để dễ join với các df khác
df_gdp = df_gdp.set_index('Nam')

# ─────────────────────────────────────────────────────────────
# KIỂM TRA KẾT QUẢ
# ─────────────────────────────────────────────────────────────
print("\n=== df_gdp SAU TIỀN XỬ LÝ ===")
print(f"Shape: {df_gdp.shape}")         # (30 năm × 9 chỉ tiêu)
print(f"\nKiểu dữ liệu:\n{df_gdp.dtypes}")
print(f"\nGiá trị thiếu (NaN):\n{df_gdp.isnull().sum()}")
print(f"\n5 hàng đầu:\n{df_gdp.head()}")
print(f"\n5 hàng cuối:\n{df_gdp.tail()}")
print(f"\nThống kê mô tả:\n{df_gdp.describe().round(2)}")

=== KIỂM TRA CẤU TRÚC GỐC ===
Shape gốc: (21, 32)

=== HEADER NĂM SAU XỬ LÝ ===
['1995', '1996', '1997', '1998', '1999', '2000', '2001', '2002', '2003', '2004', '2005', '2006', '2007', '2008', '2009', '2010', '2011', '2012', '2013', '2014', '2015', '2016', '2017', '2018', '2019', '2020', '2021', '2022', '2023', '2024']

=== df_gdp SAU TIỀN XỬ LÝ ===
Shape: (30, 9)

Kiểu dữ liệu:
GDP_TongSo              float64
TichLuyTaiSan           float64
TongTaiSanCoDinh        float64
ThayDoiTonKho           float64
TieuDungCuoiCung        float64
TieuDungNhaNuoc         float64
TieuDungHoDanCu         float64
ChanLechXuatNhapKhau    float64
SaiSo                   float64
dtype: object

Giá trị thiếu (NaN):
GDP_TongSo              0
TichLuyTaiSan           0
TongTaiSanCoDinh        0
ThayDoiTonKho           0
TieuDungCuoiCung        0
TieuDungNhaNuoc         0
TieuDungHoDanCu         0
ChanLechXuatNhapKhau    0
SaiSo                   0
dtype: int64

5 hàng đầu:
      GDP_TongSo  TichLuyTaiSan  T

In [ ]:
# ============================================================
# TIỀN XỬ LÝ DỮ LIỆU: df_gdp_region
# File: Co_cau_GDP_theo_khu_vuc_Full.xlsx
# ============================================================

import pandas as pd
import numpy as np

# ─────────────────────────────────────────────────────────────
# BƯỚC 1 – TÍCH HỢP: Đọc file, dùng hàng 0 làm header
# ─────────────────────────────────────────────────────────────
df_gdp_region = pd.read_excel(
    '/content/datasets/Co_cau_GDP_theo_khu_vuc_Full.xlsx',
    header=0    # hàng 0 là tên cột
)

print("=== CẤU TRÚC GỐC ===")
print(f"Shape: {df_gdp_region.shape}")
print(f"Columns:\n{df_gdp_region.columns.tolist()}")
print(f"\n5 hàng đầu:\n{df_gdp_region.head()}")
print(f"\n5 hàng cuối:\n{df_gdp_region.tail()}")

# ─────────────────────────────────────────────────────────────
# BƯỚC 2 – LÀM SẠCH
#   a) Đổi "So b? 2024" → 2024 trong cột Năm
#   b) Thay ".." → NaN trong cột "Giá trị: Thuế sản phẩm"
#   c) Ép kiểu cột "Giá trị: Thuế sản phẩm" về float
# ─────────────────────────────────────────────────────────────

# a) Chuẩn hóa cột Năm
df_gdp_region['Năm'] = df_gdp_region['Năm'].astype(str).str.strip()
df_gdp_region['Năm'] = df_gdp_region['Năm'].apply(
    lambda x: '2024' if 'So b' in x else x
)
df_gdp_region['Năm'] = df_gdp_region['Năm'].astype(int)

print("\n=== SAU KHI XỬ LÝ CỘT NĂM ===")
print(df_gdp_region['Năm'].tolist())

# b) & c) Xử lý cột "Giá trị: Thuế sản phẩm"
col_thue = 'Giá trị: Thuế sản phẩm'

print(f"\n=== TRƯỚC KHI LÀM SẠCH '{col_thue}' ===")
print(f"Kiểu dữ liệu: {df_gdp_region[col_thue].dtype}")
print(f"Các giá trị duy nhất: {df_gdp_region[col_thue].unique()}")

# Thay '..' → NaN
df_gdp_region[col_thue] = df_gdp_region[col_thue].replace('..', np.nan)

# Ép kiểu về float (các giá trị hợp lệ giữ nguyên, NaN giữ nguyên)
df_gdp_region[col_thue] = pd.to_numeric(df_gdp_region[col_thue], errors='coerce')

print(f"\n=== SAU KHI LÀM SẠCH '{col_thue}' ===")
print(f"Kiểu dữ liệu: {df_gdp_region[col_thue].dtype}")
print(f"Số NaN: {df_gdp_region[col_thue].isnull().sum()}")
print(df_gdp_region[['Năm', col_thue]].to_string())

# ─────────────────────────────────────────────────────────────
# BƯỚC 3 – TRÍCH LỌC: Drop các cột "Cơ cấu (%)"
# ─────────────────────────────────────────────────────────────
cols_to_drop = [
    'Cơ cấu (%): Tổng số',
    'Cơ cấu (%): Nông lâm thủy sản',
    'Cơ cấu (%): Công nghiệp xây dựng',
    'Cơ cấu (%): Dịch vụ',
    'Cơ cấu (%): Thuế sản phẩm',
]

df_gdp_region = df_gdp_region.drop(columns=cols_to_drop)

# ─────────────────────────────────────────────────────────────
# BƯỚC 4 – CHUYỂN ĐỔI: Đặt Năm làm index
# ─────────────────────────────────────────────────────────────
df_gdp_region = df_gdp_region.set_index('Năm')

# ─────────────────────────────────────────────────────────────
# KIỂM TRA KẾT QUẢ CUỐI
# ─────────────────────────────────────────────────────────────
print("\n=== df_gdp_region SAU TIỀN XỬ LÝ ===")
print(f"Shape: {df_gdp_region.shape}")
print(f"\nCác cột còn lại:\n{df_gdp_region.columns.tolist()}")
print(f"\nKiểu dữ liệu:\n{df_gdp_region.dtypes}")
print(f"\nGiá trị thiếu (NaN):\n{df_gdp_region.isnull().sum()}")
print(f"\nToàn bộ dữ liệu:\n{df_gdp_region.to_string()}")
print(f"\nThống kê mô tả:\n{df_gdp_region.describe().round(2)}")

=== CẤU TRÚC GỐC ===
Shape: (39, 11)
Columns:
['Năm', 'Giá trị: Tổng số', 'Giá trị: Nông lâm thủy sản', 'Giá trị: Công nghiệp xây dựng', 'Giá trị: Dịch vụ', 'Giá trị: Thuế sản phẩm', 'Cơ cấu (%): Tổng số', 'Cơ cấu (%): Nông lâm thủy sản', 'Cơ cấu (%): Công nghiệp xây dựng', 'Cơ cấu (%): Dịch vụ', 'Cơ cấu (%): Thuế sản phẩm']

5 hàng đầu:
    Năm  Giá trị: Tổng số  Giá trị: Nông lâm thủy sản  \
0  1986             599.0                       228.0   
1  1987            2870.0                      1164.0   
2  1988           15420.0                      7139.0   
3  1989           28093.0                     11818.0   
4  1990           41955.0                     16252.0   

   Giá trị: Công nghiệp xây dựng  Giá trị: Dịch vụ Giá trị: Thuế sản phẩm  \
0                          173.0             198.0                     ..   
1                          814.0             892.0                     ..   
2                         3695.0            4586.0                     ..   
3        

In [ ]:
# Expand mỗi giai đoạn thành các năm đơn lẻ
records = []
for _, row in df_tfp.iterrows():
    start, end = map(int, row['Giai đoạn'].split('-'))
    for year in range(start, end + 1):
        records.append({
            'Nam': year,
            'Von_pct': row['Vốn (%)'],
            'LaoDong_pct': row['Lao động (%)'],
            'TFP_pct': row['TFP (%)'],
            'TocDoTangTFP': row['Tốc độ tăng TFP']
        })

df_tfp_expanded = pd.DataFrame(records).set_index('Nam')

In [ ]:
df_tfp_expanded

,Von_pct,LaoDong_pct,TFP_pct,TocDoTangTFP
Nam,,,,
2011,50.0,15.3,34.7,2.1
2012,50.0,15.3,34.7,2.1
2013,50.0,15.3,34.7,2.1
2014,50.0,15.3,34.7,2.1
2015,50.0,15.3,34.7,2.1
2016,52.2,1.8,46.0,2.9
2017,52.2,1.8,46.0,2.9
2018,52.2,1.8,46.0,2.9
2019,52.2,1.8,46.0,2.9
